# VQ-VAE-2 for High-Fidelity Video Restoration (REDS-style)

Train a two-level VQ-VAE (VQ-VAE-2 style) as a restoration model that
maps degraded video frames to clean frames. The encoder consumes degraded frames, both top and bottom
quantizers discretize latent representations, and the decoder reconstructs the clean frame. This notebook
is a research prototype — further tuning and longer training are required for SOTA results.

**Sections:**
1. Setup & imports
2. Dataset loader + degradation pipeline
3. VQ-VAE-2 implementation (two-level)
4. Training loop (degraded -> clean)
5. Evaluation and sample visualization

In [ ]:
# Setup: imports and basic utils
import os, math, random, time
from glob import glob
from pathlib import Path
from typing import List, Tuple

import torch, torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils as vutils
from PIL import Image, ImageFilter
from einops import rearrange
from tqdm import tqdm
import numpy as np
from skimage.metrics import peak_signal_noise_ratio as compare_psnr
from skimage.metrics import structural_similarity as compare_ssim
import matplotlib.pyplot as plt

print('PyTorch version:', torch.__version__)


## Dataset loader (REDS-style) and degradation pipeline

Assumes dataset structured as `root/sequence_000/000000.png ...`. The degradation pipeline includes
downsampling (simulating low-resolution), gaussian noise, and JPEG compression simulation.

In [ ]:

class VideoFramesDataset(Dataset):
    def __init__(self, root_dir: str, seq_len: int = 7, transform=None, degrade_fn=None, use_clean_target=True):
        self.seq_dirs = sorted([str(p) for p in Path(root_dir).iterdir() if p.is_dir()])
        self.seq_len = seq_len
        self.transform = transform or transforms.Compose([transforms.Resize((256,256)), transforms.ToTensor()])
        self.degrade_fn = degrade_fn or self.default_degrade
        self.use_clean_target = use_clean_target

    def __len__(self):
        return len(self.seq_dirs)

    def load_frames(self, seq_dir: str) -> torch.Tensor:
        images = sorted(glob(os.path.join(seq_dir, "*.png")))
        if len(images) < self.seq_len:
            imgs = images + [images[-1]] * (self.seq_len - len(images))
        else:
            start = max(0, (len(images) - self.seq_len)//2)
            imgs = images[start:start+self.seq_len]
        frames = [Image.open(p).convert('RGB') for p in imgs]
        frames_t = torch.stack([self.transform(f) for f in frames], dim=0)  # [T, C, H, W]
        return frames_t

    def default_degrade(self, clean: torch.Tensor) -> torch.Tensor:
        # Simulate common degradations: downsample + gaussian noise + blur
        B = clean.shape[0]
        degraded = clean.clone()
        # per-frame operations using PIL for JPEG simulation and blur
        degraded_list = []
        for t in range(B):
            frame = transforms.ToPILImage()(clean[t].cpu())
            # random downscale
            scale = random.choice([1.0, 0.75, 0.5, 0.33])
            if scale < 1.0:
                new_size = (int(frame.width * scale), int(frame.height * scale))
                frame = frame.resize(new_size, Image.BICUBIC).resize((frame.width, frame.height), Image.BICUBIC)
            # gaussian blur sometimes
            if random.random() < 0.3:
                frame = frame.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5,1.5)))
            # JPEG-like compression: save to bytes with quality
            if random.random() < 0.5:
                from io import BytesIO
                buf = BytesIO()
                q = random.randint(30, 95)
                frame.save(buf, format='JPEG', quality=q)
                buf.seek(0)
                frame = Image.open(buf)
            tensor = self.transform(frame)
            # add gaussian noise
            tensor = tensor + 0.03 * torch.randn_like(tensor)
            tensor = tensor.clamp(0,1)
            degraded_list.append(tensor)
        degraded = torch.stack(degraded_list, dim=0)
        return degraded

    def __getitem__(self, idx):
        seq_dir = self.seq_dirs[idx]
        clean = self.load_frames(seq_dir)  # [T,C,H,W]
        degraded = self.degrade_fn(clean.clone())
        # For restoration training: input = degraded, target = clean
        return degraded.float(), clean.float()


## VQ-VAE-2 implementation (two-level hierarchical)

We implement bottom and top encoders, quantizers, and decoders, following VQ-VAE-2 design. The model
takes a degraded frame as input to encoder and reconstructs the clean frame through decoder conditioned
on top quantized features.

In [ ]:

class VectorQuantizer(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, commitment_cost=0.25):
        super().__init__()
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.commitment_cost = commitment_cost
        self.embeddings = nn.Embedding(self.num_embeddings, self.embedding_dim)
        self.embeddings.weight.data.uniform_(-1/self.num_embeddings, 1/self.num_embeddings)

    def forward(self, z):
        # z: [B, C, H, W]
        B, C, H, W = z.shape
        flatten = z.permute(0,2,3,1).contiguous().view(-1, self.embedding_dim)  # [B*H*W, C]
        # compute distances
        distances = (flatten.pow(2).sum(1, keepdim=True)
                     - 2 * flatten @ self.embeddings.weight.t()
                     + self.embeddings.weight.pow(2).sum(1, keepdim=True).t())
        encoding_indices = torch.argmin(distances, dim=1).unsqueeze(1)
        encodings = torch.zeros(encoding_indices.size(0), self.num_embeddings, device=z.device)
        encodings.scatter_(1, encoding_indices, 1)
        quantized = (encodings @ self.embeddings.weight).view(B, H, W, self.embedding_dim).permute(0,3,1,2).contiguous()
        # losses
        e_latent_loss = F.mse_loss(quantized.detach(), z)
        q_latent_loss = F.mse_loss(quantized, z.detach())
        loss = q_latent_loss + self.commitment_cost * e_latent_loss
        # straight-through
        quantized = z + (quantized - z).detach()
        avg_probs = encodings.mean(0)
        perplexity = torch.exp(- (avg_probs * torch.log(avg_probs + 1e-10)).sum())
        # return quantized, loss, perp, indices shaped [B, H, W]
        return quantized, loss, perplexity, encoding_indices.view(B, H, W)


In [ ]:

def conv_block(in_ch, out_ch, kernel=3, stride=1, padding=1):
    return nn.Sequential(nn.Conv2d(in_ch, out_ch, kernel, stride=stride, padding=padding),
                         nn.ReLU(inplace=True),
                         nn.BatchNorm2d(out_ch))

class BottomEncoder(nn.Module):
    def __init__(self, in_ch=3, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            conv_block(in_ch, 64, 4, 2, 1),   # /2
            conv_block(64, 128, 4, 2, 1),     # /4
            conv_block(128, hidden, 4, 2, 1)  # /8
        )
    def forward(self,x): return self.net(x)

class TopEncoder(nn.Module):
    def __init__(self, in_ch=128, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            conv_block(in_ch, hidden, 4, 2, 1),   # /16
            conv_block(hidden, hidden, 3, 1, 1)
        )
    def forward(self,x): return self.net(x)

class TopDecoder(nn.Module):
    def __init__(self, in_ch=64, out_ch=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(in_ch, in_ch//2, 4, 2, 1),
            nn.ReLU(),
            nn.Conv2d(in_ch//2, out_ch, 3, 1, 1)
        )
    def forward(self,x): return self.net(x)

class BottomDecoder(nn.Module):
    def __init__(self, in_ch=128+64, out_ch=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(in_ch, 128, 4, 2, 1),  # x2
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, out_ch, 4, 2, 1),
            nn.Sigmoid()
        )
    def forward(self,x): return self.net(x)


In [ ]:

class VQVAE2_Restorer(nn.Module):
    def __init__(self,
                 in_ch=3,
                 bottom_hidden=128,
                 top_hidden=256,
                 embedding_dim_top=64,
                 embedding_dim_bottom=64,
                 num_embeddings_top=512,
                 num_embeddings_bottom=512):
        super().__init__()
        # encoders operate on degraded input (for restoration)
        self.bottom_enc = BottomEncoder(in_ch=in_ch, hidden=bottom_hidden)
        self.top_enc = TopEncoder(in_ch=bottom_hidden, hidden=top_hidden)
        # top quantizer
        self.pre_vq_conv_top = nn.Conv2d(top_hidden, embedding_dim_top, 1)
        self.vq_top = VectorQuantizer(num_embeddings_top, embedding_dim_top)
        # top decoder -> conditioning for bottom
        self.top_decoder = TopDecoder(embedding_dim_top, out_ch=bottom_hidden)
        # bottom quantizer
        self.pre_vq_conv_bottom = nn.Conv2d(bottom_hidden, embedding_dim_bottom, 1)
        self.vq_bottom = VectorQuantizer(num_embeddings_bottom, embedding_dim_bottom)
        # final decoder conditioned on top decoded features
        self.decoder = BottomDecoder(in_ch=embedding_dim_bottom + bottom_hidden, out_ch=3)

    def forward(self, x):
        # x: [B,3,H,W] degraded input
        bottom = self.bottom_enc(x)    # [B, bottom_hidden, H/8, W/8]
        top = self.top_enc(bottom)     # [B, top_hidden, H/16, W/16]
        top_z = self.pre_vq_conv_top(top)
        quant_top, loss_top, ppl_top, idx_top = self.vq_top(top_z)
        # upsample top quant to bottom spatial via top_decoder
        top_cond = self.top_decoder(quant_top)  # [B, bottom_hidden, H/8, W/8]
        bottom_z = self.pre_vq_conv_bottom(bottom)
        # Optionally incorporate top_cond before bottom quantization (here bottom quant on bottom_z)
        quant_bottom, loss_bottom, ppl_bottom, idx_bottom = self.vq_bottom(bottom_z)
        # decoder input: concat quant_bottom + top_cond
        dec_in = torch.cat([quant_bottom, top_cond], dim=1)
        recon = self.decoder(dec_in)
        loss_vq = loss_top + loss_bottom
        ppl = (ppl_top + ppl_bottom) / 2.0
        return recon, loss_vq, ppl, (idx_top, idx_bottom)


## Training loop (degraded -> clean)

Loss = reconstruction MSE + VQ loss. Optionally include perceptual or LPIPS for better visual quality.

In [ ]:

def psnr_torch(pred, target):
    # pred/target in [0,1] torch
    mse = F.mse_loss(pred, target, reduction='none')
    mse = mse.mean(dim=[1,2,3])
    psnr = 10 * torch.log10(1.0 / mse)
    return psnr.mean().item()

def train_vqvae2_restorer(model: nn.Module, dataset: Dataset, device='cuda',
                          epochs=10, batch_size=8, lr=2e-4, save_dir='./vq_ckpts', val_dataset=None):
    model = model.to(device)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=4, drop_last=True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-6)
    recon_loss = nn.MSELoss()

    os.makedirs(save_dir, exist_ok=True)
    global_step = 0
    for ep in range(epochs):
        model.train()
        pbar = tqdm(loader, desc=f'Epoch {ep+1}/{epochs}')
        train_loss = 0.0
        train_psnr = 0.0
        for degraded_seq, clean_seq in pbar:
            # per-frame training flatten temporal dim
            B, T, C, H, W = degraded_seq.shape
            frames_in = rearrange(degraded_seq, 'b t c h w -> (b t) c h w').to(device)
            frames_gt = rearrange(clean_seq, 'b t c h w -> (b t) c h w').to(device)
            opt.zero_grad()
            recon, vq_loss, ppl, idxs = model(frames_in)
            loss_recon = recon_loss(recon, frames_gt)
            loss = loss_recon + vq_loss
            loss.backward()
            opt.step()
            train_loss += loss.item()
            train_psnr += psnr_torch(recon, frames_gt)
            global_step += 1
            pbar.set_postfix(loss=loss.item(), vq_loss=float(vq_loss), ppl=float(ppl))
        avg_loss = train_loss / len(loader)
        avg_psnr = train_psnr / len(loader)
        print(f'Epoch {ep+1}: Avg Loss={avg_loss:.4f}, Avg PSNR={avg_psnr:.3f} dB')
        # validation
        if val_dataset is not None:
            val_psnr, val_ssim = evaluate_model(model, val_dataset, device=device, num_samples=8)
            print(f'Validation PSNR: {val_psnr:.3f}, SSIM: {val_ssim:.3f}')
        torch.save({'model': model.state_dict(), 'opt': opt.state_dict()}, os.path.join(save_dir, f'vq_restorer_ep{ep+1}.pth'))
    return model


In [ ]:

def evaluate_model(model: nn.Module, dataset: Dataset, device='cuda', num_samples=16):
    model.eval()
    loader = DataLoader(dataset, batch_size=1, shuffle=True)
    psnrs = []
    ssims = []
    with torch.no_grad():
        for i, (degraded_seq, clean_seq) in enumerate(loader):
            if i >= num_samples: break
            B, T, C, H, W = degraded_seq.shape
            # process first frame or all frames (here evaluate middle frame)
            degraded = degraded_seq.to(device)
            frames_in = rearrange(degraded, 'b t c h w -> (b t) c h w')
            recon, _, _, _ = model(frames_in)
            recon = recon.clamp(0,1)
            # reshape to [B,T,C,H,W]
            recon_seq = rearrange(recon, '(b t) c h w -> b t c h w', b=B, t=T).cpu().numpy()
            gt_seq = clean_seq.numpy()
            # compute metrics per-frame for sequence
            for f in range(T):
                p = (recon_seq[0,f].transpose(1,2,0) * 255).astype(np.uint8)
                g = (gt_seq[0,f].transpose(1,2,0) * 255).astype(np.uint8)
                psnrs.append(compare_psnr(g, p, data_range=255))
                ssims.append(compare_ssim(g, p, multichannel=True, data_range=255))
    return float(np.mean(psnrs)), float(np.mean(ssims))


In [ ]:

def show_sample(model, dataset, device='cuda', idx=0, save_path=None):
    model.eval()
    degraded_seq, clean_seq = dataset[idx]
    B = 1
    degraded = degraded_seq.unsqueeze(0).to(device)  # [1,T,C,H,W]
    frames_in = rearrange(degraded, 'b t c h w -> (b t) c h w')
    with torch.no_grad():
        recon, _, _, _ = model(frames_in)
    recon = recon.clamp(0,1).cpu()
    recon_seq = rearrange(recon, '(b t) c h w -> b t c h w', b=B, t=degraded.shape[1])
    # display middle frame
    mid = recon_seq[0, recon_seq.shape[1]//2]
    gt_mid = clean_seq[recon_seq.shape[1]//2]
    degraded_mid = degraded_seq[recon_seq.shape[1]//2]
    fig, axs = plt.subplots(1,3, figsize=(12,4))
    axs[0].imshow(np.transpose(degraded_mid.numpy(), (1,2,0))); axs[0].set_title('Degraded')
    axs[1].imshow(np.transpose(mid.numpy(), (1,2,0))); axs[1].set_title('Reconstruction')
    axs[2].imshow(np.transpose(gt_mid.numpy(), (1,2,0))); axs[2].set_title('Ground Truth')
    for ax in axs: ax.axis('off')
    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
    plt.show()


## Example usage

Update `DATA_ROOT` and `VAL_ROOT` to your dataset paths. Then run the cells below to train and visualize.

`DATA_ROOT` should point to training folder with subfolders per sequence. For quick tests, use a small subset.

In [ ]:

# Example configuration (edit paths before running)
DATA_ROOT = '/path/to/REDS/train'   # <-- set this
VAL_ROOT = '/path/to/REDS/val'      # <-- set this
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# instantiate dataset and model for a quick run (use small batch and epochs for tests)
train_ds = VideoFramesDataset(DATA_ROOT, seq_len=7, transform=transforms.Compose([transforms.Resize((256,256)), transforms.ToTensor()]))
val_ds = VideoFramesDataset(VAL_ROOT, seq_len=7, transform=transforms.Compose([transforms.Resize((256,256)), transforms.ToTensor()]))

model = VQVAE2_Restorer(in_ch=3, bottom_hidden=128, top_hidden=256,
                        embedding_dim_top=64, embedding_dim_bottom=64,
                        num_embeddings_top=512, num_embeddings_bottom=512)

# For a quick smoke test, uncomment following lines and run for 1 epoch:
# trained_model = train_vqvae2_restorer(model, train_ds, device=DEVICE, epochs=1, batch_size=4, lr=2e-4, save_dir='./vq_ckpts', val_dataset=val_ds)
# show_sample(trained_model, val_ds, device=DEVICE, idx=0)
print('Configure DATA_ROOT and VAL_ROOT and then run the training cells.')